# Notebook 03: Agent Testing & Research Report

**Goal:** Run the full multi-agent pipeline interactively. Evaluate output quality. Iterate on prompts.

**Prerequisites:** Run notebook 02 first to populate the vector store with at least one ticker.

In [ ]:
import sys
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from IPython.display import Markdown, display

TICKER = 'RELIANCE.NS'  # Change to any ingested ticker
print(f'Testing agents for: {TICKER}')

## 1. Quick Q&A (no full pipeline)

In [ ]:
from src.agents.orchestrator import ResearchOrchestrator

orch = ResearchOrchestrator()

# Test: quick Q&A
questions = [
    f'What is the debt-to-equity ratio of {TICKER}?',
    f'What is the revenue growth trend for {TICKER}?',
    f'What are the main risks for {TICKER}?',
]

for q in questions:
    print(f'\nQ: {q}')
    answer = orch.quick_answer(TICKER, q)
    print(f'A: {answer[:300]}...')

## 2. Individual Agent Tests

In [ ]:
from src.agents.fundamental_agent import FundamentalAgent
from src.rag.retriever import Retriever
from config.nifty50_tickers import NIFTY50_TICKERS

nse_symbol = TICKER.replace('.NS', '')
company_name = NIFTY50_TICKERS.get(nse_symbol, nse_symbol)

retriever = Retriever()
agent = FundamentalAgent(retriever=retriever)

print('Running Fundamental Agent...')
result = agent.run(TICKER, company_name)
display(Markdown(result))

In [ ]:
from src.agents.sentiment_agent import SentimentAgent

sentiment_agent = SentimentAgent(retriever=retriever)
print('Running Sentiment Agent...')
sentiment = sentiment_agent.run(TICKER, company_name)
display(Markdown(sentiment))

## 3. Full Research Report

In [ ]:
# This runs all 4 agents in sequence (~2-4 minutes)
# Make sure your ANTHROPIC_API_KEY is set

print(f'Generating full research report for {TICKER}...')
print('This will take 2-4 minutes...\n')

report = orch.research(
    ticker_yf=TICKER,
    horizon='3 years',
    save=True,
)

print('Report generated!')
if report.metadata.get('saved_to'):
    print(f"Saved to: {report.metadata['saved_to']}")

In [ ]:
# Display conviction summary (the retail-investor friendly part)
display(Markdown('## Conviction Summary (Plain English)'))
display(Markdown(report.conviction_summary))

In [ ]:
# Display scenario analysis
display(Markdown('## Scenario Analysis (Bull / Base / Bear)'))
display(Markdown(report.scenario_analysis))

## 4. Evaluate Retrieval Quality

Test: for each question, does the retriever return the right chunk?

In [ ]:
eval_cases = [
    ('What is the revenue?', 'income_statement'),
    ('What is the debt?', 'balance_sheet'),
    ('What is the free cash flow?', 'cash_flow'),
    ('What did management say?', 'news'),
    ('What is the PE ratio?', 'key_ratios'),
]

print('Retrieval Evaluation')
print('='*60)
correct = 0
for query, expected_section in eval_cases:
    results = retriever.retrieve(query, ticker=TICKER, top_k=1)
    if results:
        got_section = results[0]['metadata']['section']
        match = '✓' if got_section == expected_section else '✗'
        if got_section == expected_section:
            correct += 1
        print(f'{match} Query: "{query}"')
        print(f'  Expected: {expected_section} | Got: {got_section} | Score: {results[0]["score"]:.3f}')

print(f'\nRetrieval Accuracy: {correct}/{len(eval_cases)} = {correct/len(eval_cases)*100:.0f}%')